In [34]:
import warnings
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import scipy.sparse


warnings.filterwarnings("ignore")

In [35]:
from preprocessing import preprocess_data
from basic_models import random_forest_classifier
from basic_models import gradient_boosting_classifier
from basic_models import LSTM_preds
from basic_models import logistic_reg_model

In [36]:
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [37]:
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, DotProduct, WhiteKernel


In [38]:
data, X_train_processed, X_test_processed, y_train, y_test = preprocess_data("stores_sales_forecasting 2.csv")
data.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,lag_profit_1,rolling_sales_mean_3,rolling_profit_mean_3,cohort,churn,Order Year,Order Month,Ship Year,Ship Month,Shipping Delay
20,79,US-2014-147606,2014-11-26,2014-12-01,Second Class,JE-15745,Joel Eaton,Consumer,United States,Houston,...,1.2130,316.092000,-42.551067,0,0,2014,11,2014,12,5
39,178,US-2015-101511,2015-11-21,2015-11-23,Second Class,JE-15745,Joel Eaton,Consumer,United States,Newark,...,-14.4750,171.047333,-8.199733,12,0,2015,11,2015,11,2
51,235,US-2017-100930,2017-04-07,2017-04-12,Standard Class,CS-12400,Christopher Schild,Home Office,United States,Tampa,...,-248.2458,370.848833,-116.764600,0,1,2017,4,2017,4,5
54,242,CA-2016-157749,2016-06-04,2016-06-09,Second Class,KL-16645,Ken Lonsdale,Consumer,United States,Chicago,...,-4.6752,202.864333,-160.638733,23,1,2016,6,2016,6,5
55,243,CA-2016-157749,2016-06-04,2016-06-09,Second Class,KL-16645,Ken Lonsdale,Consumer,United States,Chicago,...,-120.5130,64.319000,-42.673000,23,1,2016,6,2016,6,5


In [39]:
X_train_dense = X_train_processed.toarray() if scipy.sparse.issparse(X_train_processed) else X_train_processed
X_test_dense = X_test_processed.toarray() if scipy.sparse.issparse(X_test_processed) else X_test_processed

lstm_out = LSTM_preds(X_train_dense, X_test_dense, y_train)

Epoch 1/20
11/11 [==============================] - 3s 14ms/step - loss: 0.5682 - accuracy: 0.0000e+00
Epoch 2/20
11/11 [==============================] - 0s 15ms/step - loss: 0.0158 - accuracy: 0.0000e+00
Epoch 3/20
11/11 [==============================] - 0s 15ms/step - loss: -0.4477 - accuracy: 0.0000e+00
Epoch 4/20
11/11 [==============================] - 0s 15ms/step - loss: -0.8326 - accuracy: 0.0000e+00
Epoch 5/20
11/11 [==============================] - 0s 15ms/step - loss: -1.2257 - accuracy: 0.0000e+00
Epoch 6/20
11/11 [==============================] - 0s 15ms/step - loss: -1.4891 - accuracy: 0.0000e+00
Epoch 7/20
11/11 [==============================] - 0s 15ms/step - loss: -1.8231 - accuracy: 0.0000e+00
Epoch 8/20
11/11 [==============================] - 0s 15ms/step - loss: -2.0914 - accuracy: 0.0000e+00
Epoch 9/20
11/11 [==============================] - 0s 15ms/step - loss: -2.4635 - accuracy: 0.0000e+00
Epoch 10/20
11/11 [==============================] - 0s 15ms/step 

In [34]:
# Define the kernel: Constant * RBF
kernel = C(1.0) * RBF(length_scale=1.0)

gp_model = GaussianProcessClassifier(kernel=kernel, random_state=42))

In [35]:
gp_model.fit(X_train_dense, y_train)

GaussianProcessClassifier(kernel=1**2 * RBF(length_scale=1), random_state=42)

In [36]:
y_pred = gp_model.predict(X_test_dense)

In [37]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7126


## Kernels: RBF + Periodic + Linear Kernel

In [47]:
# Define individual kernels
rbf_kernel = RBF(length_scale=1.0)  # Captures smooth trends
periodic_kernel = ExpSineSquared(length_scale=1.0, periodicity=3.0)  # Models periodic patterns
linear_kernel = DotProduct(sigma_0=1.0)  # Models linear relationships


In [48]:
# Add WhiteKernel (like alpha in regression)
# noise_kernel = WhiteKernel(noise_level=1e-5)

In [49]:
# Combine them using addition
combined_kernel = rbf_kernel + periodic_kernel + linear_kernel

In [50]:
# Initialize the Gaussian Process Classifier
gp = GaussianProcessClassifier(kernel=combined_kernel, random_state=42)

In [54]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train_dense)
# X_test_scaled = scaler.transform(X_test_dense)

In [55]:
from sklearn.decomposition import PCA

pca = PCA(n_components=50)  # Keep 50 principal components
X_train_pca = pca.fit_transform(X_train_dense)
X_test_pca = pca.transform(X_test_dense)

In [56]:
gp.fit(X_train_scaled, y_train)

LinAlgError: 477-th leading minor of the array is not positive definite

In [30]:
y_pred = gp_model.predict(X_test_dense)

AttributeError: '_BinaryGaussianProcessClassifierLaplace' object has no attribute 'pi_'